# Pharmaceutical Document RAG

A high-performance pharmaceutical document question-answering system built on `RAGPipeline`.

---

| Component | Details |
|---|---|
| **Embedding** | `sentence-transformers/all-MiniLM-L6-v2` (GPU-accelerated) |
| **Chunking** | Fixed-size (512 tokens, 50 overlap) |
| **Retrieval** | Hybrid — Vector + BM25, reciprocal rerank (top-3) |
| **LLM** | Mistral 7B Instruct (local, 4K context) |
| **OCR** | Tesseract (parallel, 200 DPI) |
| **UI** | Gradio |
| **Performance** | Index persistence + batched classification |

## 1. Install Dependencies

In [21]:
%pip install -q pymupdf
%pip install -q llama-index llama-index-core
%pip install -q llama-index-embeddings-huggingface
%pip install -q llama-index-llms-llama-cpp
%pip install -q llama-index-retrievers-bm25
%pip install -q llama-cpp-python
%pip install -q sentence-transformers huggingface-hub torch
%pip install -q pytesseract pillow
%pip install -q "gradio>=6.9.0" --upgrade
%pip install -q "nest-asyncio>=1.6.0"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
from pathlib import Path

# Ensure the src directory is on the path so rag can be imported
sys.path.insert(0, str(Path("..").resolve() / "src"))

from rag import RAGPipeline
import gradio as gr

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 3. Initialize RAG Pipeline

Loads the GGUF model directly from disk — no download needed.

> **GPU offload:** `n_gpu_layers=-1` offloads all layers to GPU (CUDA). Set `n_gpu_layers=0` to run on CPU only.

> **Index persistence:** `persist_dir="./storage"` saves the index to disk after building, enabling instant loading on subsequent runs.

In [23]:
MODEL_PATH = r"C:\LLM Models\Mistral\mistral-7b-instruct-v0.2.Q4_K_M.gguf"

rag = RAGPipeline(model_path=MODEL_PATH, persist_dir="./storage")
print("RAGPipeline initialized.")

llama_context: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://hu

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HT

RAGPipeline initialized.


## 4. Gradio UI

Upload a pharmaceutical PDF, click **Build Pipeline** to index it, then ask questions in the chat. Each answer includes source citations and per-chunk confidence scores.

**Performance optimizations:**
- GPU-accelerated embeddings for 5-10x faster indexing
- Fixed-size chunking (20-50x faster than semantic)
- Index persistence (instant loading after first build)
- Parallel OCR processing (3-5x faster for scanned pages)
- Reduced LLM context window for faster inference

In [ ]:
_pipeline_ready = False
_processing_queue = []
_current_file_index = 0


def build_pipeline(pdf_files, classify_docs):
    """Index uploaded PDFs (single or multiple) and mark the pipeline as ready."""
    global _pipeline_ready, _processing_queue, _current_file_index
    
    if pdf_files is None or (isinstance(pdf_files, list) and len(pdf_files) == 0):
        return "No files uploaded.", "*Upload one or more PDFs and click **Build Pipeline**.*"
    
    # Handle both single file and multiple files
    if not isinstance(pdf_files, list):
        pdf_files = [pdf_files]
    
    try:
        _pipeline_ready = False
        _processing_queue = pdf_files
        _current_file_index = 0
        
        num_files = len(pdf_files)
        
        # Progress callback to track file processing
        progress_updates = []
        def progress_callback(current, total, filename):
            progress_updates.append(f"✓ Loaded {current}/{total}: {filename}")
        
        # Build from single or multiple PDFs
        if num_files == 1:
            status_msg = f"Processing: {os.path.basename(pdf_files[0])}"
            rag.build(pdf_files[0], classify_docs=classify_docs)
            file_label = os.path.basename(pdf_files[0])
        else:
            status_msg = f"Processing {num_files} files..."
            # Extract file paths from the uploaded files
            file_paths = [f.name if hasattr(f, 'name') else f for f in pdf_files]
            rag.build_from_multiple_pdfs(
                file_paths, 
                classify_docs=classify_docs,
                progress_callback=progress_callback
            )
            file_label = f"{num_files} files"
        
        _pipeline_ready = True
        
        if classify_docs:
            file_label += " (doc types classified)"
        status = f"✓ Ready — {file_label}"

        # Get statistics
        stats = rag.get_stats()
        stats_md = (
            f"| Stat | Value |\n"
            f"|---|---|\n"
            f"| Files | {stats.get('total_files', 1)} |\n"
            f"| Pages | {stats['total_pages']} |\n"
            f"| Chunks | {stats['total_chunks']} |\n"
        )
        
        # Show file names if multiple files
        if stats.get('total_files', 1) > 1:
            file_list = "\n".join(f"- {name}" for name in stats.get('file_names', []))
            stats_md += f"\n**Indexed files:**\n\n{file_list}\n\n"
        
        if stats["classified"] and stats["doc_type_counts"]:
            type_rows = "\n".join(
                f"| &nbsp;&nbsp;`{dt}` | {count} pages |"
                for dt, count in sorted(stats["doc_type_counts"].items())
            )
            stats_md += f"| **Document types** | |\n{type_rows}\n"
        else:
            stats_md += "| Document types | *(enable classification to see)* |\n"
        
        # Append progress updates if multiple files
        if progress_updates:
            stats_md += f"\n\n**Processing log:**\n\n" + "\n".join(f"- {p}" for p in progress_updates)

        return status, stats_md
    except Exception as exc:
        import traceback
        error_details = traceback.format_exc()
        return f"Error: {exc}", f"```\n{error_details}\n```"


def ask(question, history, classify):
    """Run a RAG query and return updated chat history + formatted sources."""
    if not question.strip():
        return history, "", "*Ask a question above.*"

    if not _pipeline_ready:
        history = history + [
            {"role": "user", "content": question},
            {"role": "assistant", "content": "Please upload and build a document first."},
        ]
        return history, "", "*No sources — pipeline not ready.*"

    try:
        result = rag.query_with_sources(question, classify=classify)
    except Exception as exc:
        history = history + [
            {"role": "user", "content": question},
            {"role": "assistant", "content": f"Error: {exc}"},
        ]
        return history, "", "*Error during retrieval.*"

    answer = result["answer"]
    sources = result["sources"]
    chunk_count = result["chunk_count"]
    query_category = result["query_category"]

    # Build sources panel
    sources_md = f"**{chunk_count} chunk(s) retrieved**"
    if query_category:
        sources_md += f" &nbsp;·&nbsp; Query classified as: **`{query_category}`**"
    sources_md += "\n\n---\n\n"

    for i, src in enumerate(sources, 1):
        confidence_str = f"{src['score']:.1f}%" if src["score"] is not None else "N/A"
        pharma_label = src.get("pharma_doc_type", "unknown")
        sources_md += (
            f"**Source {i}** &nbsp;·&nbsp; "
            f"`{src['file']}` &nbsp;·&nbsp; "
            f"Page **{src['page']}** &nbsp;·&nbsp; "
            f"Confidence: **{confidence_str}** &nbsp;·&nbsp; "
            f"Doc type: {src['doc_type']} &nbsp;·&nbsp; "
            f"Pharma type: **{pharma_label}**\n\n"
            f"> {src['text'][:300]}{'...' if len(src['text']) > 300 else ''}\n\n"
            f"---\n\n"
        )

    history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    return history, "", sources_md


# ── Layout ────────────────────────────────────────────────────────────────────
with gr.Blocks(title="Pharma RAG") as demo:

    gr.Markdown("# Pharmaceutical Document RAG\nUpload a PDF, build the pipeline, then ask questions.")

    # Document ingestion row
    with gr.Row():
        pdf_input = gr.File(
            label="Upload PDF(s)", 
            file_types=[".pdf"], 
            file_count="multiple",
            scale=3
        )
        with gr.Column(scale=2):
            classify_docs_toggle = gr.Checkbox(
                label="Classify document pages by pharma type",
                value=False,
                info="Enables targeted retrieval per document category. Adds one LLM call per page.",
            )
            build_btn = gr.Button("Build Pipeline", variant="primary", size="lg")
            status_box = gr.Textbox(
                label="Status",
                value="No document loaded.",
                interactive=False,
            )

    # Document stats panel
    with gr.Accordion("Document Stats", open=True):
        stats_display = gr.Markdown("*Stats will appear here after building the pipeline.*")

    gr.Markdown("---")

    chatbot = gr.Chatbot(label="Chat", height=420)

    with gr.Row():
        question_input = gr.Textbox(
            label="Question",
            placeholder="e.g. What are the storage conditions?",
            scale=5,
        )
        ask_btn = gr.Button("Ask", variant="primary", scale=1)

    classify_query_toggle = gr.Checkbox(
        label="Classify query to restrict retrieval to matching document type",
        value=False,
        info="Uses the LLM to detect which pharma doc type best answers the query, then filters chunks accordingly.",
    )

    # Sources panel
    with gr.Accordion("Sources & Confidence", open=True):
        sources_display = gr.Markdown("*Sources will appear here after asking a question.*")

    # Event wiring
    build_btn.click(build_pipeline, inputs=[pdf_input, classify_docs_toggle], outputs=[status_box, stats_display])
    ask_btn.click(
        ask,
        inputs=[question_input, chatbot, classify_query_toggle],
        outputs=[chatbot, question_input, sources_display],
    )
    question_input.submit(
        ask,
        inputs=[question_input, chatbot, classify_query_toggle],
        outputs=[chatbot, question_input, sources_display],
    )

demo.launch(share=False, theme=gr.themes.Soft())

INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7865


INFO:httpx:HTTP Request: GET http://127.0.0.1:7865/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7865/ "HTTP/1.1 200 OK"


* To create a public link, set `share=True` in `launch()`.


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"
INFO:rag_pipeline:Loading persisted index from ./storage...
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:rag_pipeline:Loaded index with 20 chunks.
DEBUG:bm25s:Building index from IDs objects
INFO:rag_pipeline:Hybrid retriever ready.
INFO:rag_pipeline:RAG pipeline ready.
INFO:rag_pipeline:Loading persisted index from ./storage...
INFO:llama_index.core.indices.loading:Loading all indices.
INFO:rag_pipeline:Loaded index with 20 chunks.
DEBUG:bm25s:Building index from IDs objects
INFO:rag_pipeline:Hybrid retriever ready.
INFO:rag_pipeline:RAG pipeline ready.
INFO:rag_pipeline:Query classified as: packaging_specification
DEBUG:bm25s:Building index from IDs objects
